# Musanga V4 — NB05 V4.001 | Harmonized modality experiment (Option B)

**Purpose:** isolate the added value of structural (DSM) and multispectral (MS) information for Musanga crown segmentation and fractional-cover estimation. This is a *new, independent experiment*; NB02/NB03/NB04 checkpoints and outputs remain immutable.

## Pre-registered primary comparison

| Arm | Features | Backbone and initialization | Validation |
|---|---|---|---|
| M0 | RGB (3 channels) | ResNet18 U-Net, frozen NB02 V4.007 baseline | Original NB01 spatial folds |
| M1 | RGB + DSM (same predefined structural feature stack) | **Same ResNet18 U-Net** initialized from NB02; added input weights zero | Same outer/inner folds, loss, optimization, stopping |
| M2 | RGB + MS (red, green, red-edge, NIR) | **Same ResNet18 U-Net** initialized from NB02; added input weights zero | Same outer/inner folds, loss, optimization, stopping |

**Interpretation:** M0 is the saved RGB backbone, held fixed for direct pairing. For an entirely symmetric *from-scratch training* comparison, M0 must be retrained with exactly the same recipe (a later, separately labeled arm); do not claim identical original optimizer settings without verifying NB02.

**Primary estimand:** paired change in *plot-level* Musanga fractional-cover absolute error (percentage points), calculated from full original reference-unit spatial hold-outs on the **identical RGB–DSM–MS valid footprint**. Report R², RMSE, bias, F1/IoU and MOS high-cover (>40%) separately. Also report RGB–DSM and RGB–MS on their larger respective matched footprints, not as direct three-way comparisons.

**Fold definition:** the immutable NB01 spatial CV assignment of the same 78 field plots and 34 complete circles, with exclusion rules and spatial guards from NB01. Each eligible reference unit is predicted once out of fold. Three folds are a computationally tractable *partition of spatial held-out units*, not three scientific replicates, and not leave-one-entire-site-out.

**Yoko:** include the 2020 Carlier P1/P2 MS coverage in *the same folds as every other site*. A valid training fold may contain Yoko labelled chips **only if Yoko is in the outer-training partition**; held-out Yoko labels must never leak into training. P1 and P2 must not be spliced band-by-band into a single observation. Missing MS coverage is missing data, not negative Musanga.

## Stage 1 — QA and frozen experimental contract (this notebook)

Run **00 → 01 → 02 → 03**. These cells inspect raster headers and reference geometry only, and produce the *candidate* modality/plot support contract. **They do not train models**. Review the Yoko filename/band matches, footprint overlap, and crown-scale coregistration before permitting expensive Stage 2 fitting.

The subsequent trainer must use the same spatial manifests, sampling seeds, augmentation, batch sampler, 256-pixel windows, training loss, inner-site-macro BCE selection and fixed 0.5 prediction threshold for M1 and M2, with frozen RGB as an eligible fallback.


## 00 — Read-only environment and immutable reference contract


In [ ]:
from pathlib import Path
import os, re, json, hashlib
from collections import defaultdict
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from shapely.geometry import box
from rasterio.warp import transform_bounds
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
DATA=ROOT/'Data'
REF=ROOT/'Outputs/v4/01_reference_field78_combined'
N0=ROOT/'Outputs/v4/00_input_audit'
OUT=ROOT/'Outputs/v4/05_harmonized_optionB_v4_001'
OUT.mkdir(parents=True,exist_ok=True)
CONTRACT=json.loads((REF/'01_field78_reference_contract.json').read_text())
NB00=json.loads(Path(CONTRACT['input_manifest']).read_text())
FOLDS=pd.read_csv(CONTRACT['folds_csv'])
CF=pd.read_csv(REF/'01_field78_circle_folds.csv')
PLOTS=gpd.read_file(NB00['field_support_qa']).to_crs('EPSG:32635')
CIRCLES=gpd.read_file(NB00['circle_complete_roi']).to_crs('EPSG:32635')
assert len(PLOTS)==78 and len(CIRCLES)==34
assert FOLDS.source_id.is_unique and set(FOLDS.cv_fold)=={0,1,2}
assert set(CF.cv_fold)=={0,1,2} and len(CF)==34
PLOTS=PLOTS.merge(FOLDS[['source_id','site_key','cv_fold']],left_on='Plot_name',
                  right_on='source_id',validate='one_to_one')
CIRCLES['circle_id']=np.arange(len(CIRCLES))
CIRCLES=CIRCLES.merge(CF[['circle_id','site_key','cv_fold',
    'cross_fold_plot_intersection']],on='circle_id',validate='one_to_one')
ROI=gpd.read_file(DATA/'Ortho_RGB/ROI.shp').to_crs('EPSG:32635')
CV=pd.read_csv(CONTRACT['cv_manifest'])
assert not CV.duplicated(['cv_fold','mosaic','row','col','width','height']).any()
assert set(CV.cv_fold)=={0,1,2}
print('Frozen plot counts by site/fold')
display(PLOTS.groupby(['site_key','cv_fold']).size().unstack(fill_value=0))
print('Frozen circles by site/fold')
display(CIRCLES.groupby(['site_key','cv_fold']).size().unstack(fill_value=0))
print('Frozen window manifest',len(CV),'rows')


## 01 — Sensor registry including Yoko P1/P2 (headers only)


In [ ]:
# Historical exact RGB–DSM pairing is read from NB00; do not guess DSM filenames.
PAIRS=pd.read_csv(N0/'00_rgb_dsm_pair_audit.csv')
assert len(PAIRS)==6 and PAIRS.rgb.is_unique
RGB_DIR=DATA/'Ortho_RGB'
REG=pd.read_csv(N0/'00_ms_raster_headers.csv')
REG=REG.loc[REG.error.fillna('').eq('')]
MS_BANDS={}
RGB_TO_SITE={}
for record in PAIRS.itertuples(index=False):
    p=Path(record.rgb_path)
    assert p.is_file() and Path(record.dsm_path).is_file()
    RGB_TO_SITE[p.name]=str(next(iter(CV.loc[CV.mosaic.eq(p.name),'site'].unique())))
assert len(RGB_TO_SITE)==6
# Preserve exact historical four-band selection; P1/P2 are added separately.
BASE_GROUPS=['Baego_MS','MOS_1_2_MS','MOS_3_4_MS','MOS_5_6_MS',
             'Yangambi_YBJoseph_right_MS','Yangambi_YBForgeo_right_MS']
MOSAIC_KEY={
'Baego_MS':'Baego.tif',
'MOS_1_2_MS':'Mos_1_2_transparent_mosaic_group1_georef.tif',
'MOS_3_4_MS':'Mos_3_4_transparent_mosaic_group1_georef.tif',
'MOS_5_6_MS':'Mos_5_6_transparent_mosaic_group1_georef.tif',
'Yangambi_YBJoseph_right_MS':'Yangambi_georef.tif',
'Yangambi_YBForgeo_right_MS':'Yangambi_georef.tif'}
for group in BASE_GROUPS:
    part=REG[REG.group.eq(group)]
    if group=='Baego_MS':
        assert len(part)==1,(group,len(part))
        p=Path(part.iloc[0].path)
        with rasterio.open(p) as src:
            assert src.count>=4
        MS_BANDS[group]={b:p for b in ('red','green','red_edge','nir')}
    else:
        files={}
        for b in ('red','green','red_edge','nir'):
            matches=part.loc[part.band_hint.eq(b)]
            if b=='red_edge' and group=='MOS_1_2_MS' and len(matches)>1:
                matches=matches.loc[matches.name.eq(
                    'mlt_Mos_1_2_transparent_mosaic_red edge_georef.tif')]
            assert len(matches)==1,(group,b,list(matches.name))
            files[b]=Path(matches.iloc[0].path)
        MS_BANDS[group]=files
# Enumerate only the new Yoko names, with no image pixel reads.
yoko_root=DATA
yoko_folders=[]
for folder,dirs,files in os.walk(yoko_root):
    for d in dirs:
        if 'carlier2020_yoko_micasense' in d.lower():
            yoko_folders.append(Path(folder)/d)
    dirs[:]=[d for d in dirs if d not in
        {'Outputs','.git','__pycache__'}]
# Folders may also be dataset filename prefixes rather than directory names.
raw=[]
for p in yoko_folders:
    raw.extend(q for q in p.rglob('*') if q.is_file() and q.suffix.lower() in ('.tif','.tiff'))
# Support plain TIFF filenames stored directly in any DATA subdirectory.
for folder,dirs,files in os.walk(yoko_root):
    for name in files:
        if 'carlier2020_yoko_micasense' in name.lower() and
           name.lower().endswith(('.tif','.tiff')):
            raw.append(Path(folder)/name)
    dirs[:]=[d for d in dirs if d not in {'Outputs','.git','__pycache__'}]
raw=sorted(set(raw),key=str)
def classify(name):
    s=re.sub('[^a-z0-9]+','_',name.lower())
    if re.search(r'(^|_)(red_edge|rededge|red_?edge)(_|$)',s):return 'red_edge'
    if re.search(r'(^|_)(nir|near_infrared)(_|$)',s):return 'nir'
    if re.search(r'(^|_)green(_|$)',s):return 'green'
    if re.search(r'(^|_)red(_|$)',s):return 'red'
    return None
YOKO={}
for flight in ('P1','P2'):
    matches=[]
    for p in raw:
        s=str(p).lower()
        if re.search(r'(^|[_/\\-])'+flight.lower()+r'($|[_/\\.-])',s):
            # filename first, then parent band-labeled folder
            band=classify(p.stem) or classify(p.parent.name)
            if band:matches.append((band,p))
    found=defaultdict(list)
    for band,p in matches:found[band].append(p)
    for band in ('red','green','red_edge','nir'):
        if len(found[band])!=1:
            print('YOKO',flight,band,'candidates',list(map(str,found[band])))
    if all(len(found[b])==1 for b in ('red','green','red_edge','nir')):
        group=f'Yoko_Carlier2020_MicaSense_{flight}_MS'
        YOKO[group]={b:found[b][0] for b in ('red','green','red_edge','nir')}
        yoko_rgb=[p for p in RGB_DIR.glob('*.tif') if 'yoko' in p.name.lower()]
        assert len(yoko_rgb)==1,('Yoko RGB mosaic ambiguity',yoko_rgb)
        MOSAIC_KEY[group]=yoko_rgb[0].name
        MS_BANDS[group]=YOKO[group]
    else:
        print('YOKO FLIGHT NOT REGISTERED',flight,'manual filename validation required')
print('DISCOVERED YOKO SPECTRAL FILES:',len(raw))
display(pd.DataFrame([dict(flight=k,band=b,path=str(p)) for k,ff in YOKO.items()
                     for b,p in ff.items()]))
assert len(YOKO)==2,'Need 4 unambiguous first-band TIFFs for each P1 and P2 before training'
HEADERS=[]
for group,files in MS_BANDS.items():
    for band,p in files.items():
        with rasterio.open(p) as src:
            assert src.count>=1 and src.crs is not None,(group,band,p)
            index=('red','green','red_edge','nir').index(band)+1 if group=='Baego_MS' else 1
            assert src.count>=index
            HEADERS.append(dict(group=group,band=band,filename=p.name,
                 data_band=index,path=str(p),crs=str(src.crs),width=src.width,
                 height=src.height,bounds=str(src.bounds),pixel_m=abs(src.transform.a)))
HEADERS=pd.DataFrame(HEADERS)
HEADERS.to_csv(OUT/'05_harmonized_sensor_registry.csv',index=False)
display(HEADERS[['group','band','filename','data_band','pixel_m']])


## 02 — Pairwise and common-support candidate manifest


In [ ]:
# BOUNDING BOX overlap is a candidate only. True MS-valid, DSM-valid pixel
# footprints must be evaluated in later bounded window QA before training.
DSM_BY_RGB={Path(z.rgb_path).name:Path(z.dsm_path) for z in PAIRS.itertuples(index=False)}
U=[]
for kind,frame,idcol in [('plot',PLOTS,'Plot_name'),('circle',CIRCLES,'circle_id')]:
    for q in frame.itertuples(index=False):
        if kind=='circle' and bool(q.cross_fold_plot_intersection):continue
        geo=q.geometry
        if geo is None or geo.is_empty:continue
        possible=[m for m in set(MOSAIC_KEY.values()) if m in DSM_BY_RGB
                  and m in RGB_TO_SITE and RGB_TO_SITE[m]==q.site_key]
        for mosaic in possible:
            with rasterio.open(RGB_DIR/mosaic) as ds, rasterio.open(DSM_BY_RGB[mosaic]) as sd:
                geometry=geo.intersection(box(*ds.bounds))
                if geometry.is_empty:continue
                rgeom=box(*transform_bounds(sd.crs,ds.crs,*sd.bounds))
                dsm_area=geometry.intersection(rgeom).area
                acquisitions=[g for g,m in MOSAIC_KEY.items() if m==mosaic]
                footprints=[]
                for group in acquisitions:
                    bboxes=[]
                    for path in set(MS_BANDS[group].values()):
                        with rasterio.open(path) as band:
                            bboxes.append(box(*transform_bounds(
                                band.crs,ds.crs,*band.bounds,densify_pts=21)))
                    # intersection of ALL FOUR band bounding boxes, not union
                    intersection=geometry
                    for b in bboxes:intersection=intersection.intersection(b)
                    footprints.append(intersection)
                from shapely.ops import unary_union
                ms_region=unary_union(footprints) if footprints else geometry.intersection(box(0,0,0,0))
                common=ms_region.intersection(rgeom)
                U.append(dict(kind=kind,unit_id=str(getattr(q,idcol)),site=q.site_key,
                     fold=int(q.cv_fold),mosaic=mosaic,plot_area_m2=geo.area,
                     rgb_bbox_area_m2=geometry.area,dsm_bbox_area_m2=dsm_area,
                     ms_bbox_area_m2=ms_region.area,
                     common_bbox_area_m2=common.area,
                     common_bbox_fraction=common.area/geo.area))
UNIT_CANDIDATES=pd.DataFrame(U)
assert not UNIT_CANDIDATES.empty
UNIT_CANDIDATES.to_csv(OUT/'05_candidate_unit_modality_support.csv',index=False)
print('Candidate unit overlap, NOT confirmed valid-pixel support:')
display(UNIT_CANDIDATES.groupby(['site','fold','kind']).agg(
    n_units=('unit_id','nunique'),
    n_common_candidate=('common_bbox_area_m2',lambda x:int((x>0).sum())),
    median_common_bbox_frac=('common_bbox_fraction','median')).round(3))


## 03 — Freeze the Option B protocol and stop before expensive fitting


In [ ]:
# Stage 1 output is a provenance-audited *candidate* contract, not a final
# scientific validation mask. No global image reads or new model training.
import datetime
protocol=dict(
  name='NB05 Option B harmonized modality comparison',
  status='STAGE1_CANDIDATE_SUPPORT_NOT_PIXEL_VALIDATED',
  frozen_cv_manifest=str(CONTRACT['cv_manifest']),
  frozen_plot_folds=str(CONTRACT['folds_csv']),
  frozen_circle_folds=str(REF/'01_field78_circle_folds.csv'),
  dataset_sites=['Baego','MOS','Yangambi','Yoko'],
  outer_folds=[0,1,2],tile_px=256,holdout_unit='original plot/circle',
  primary_pairing='identical RGB+DSM+four-band MS valid pixels per unit',
  mandatory_training=['same ResNet18 U-Net backbone',
      'same original NB02 initialized RGB encoder and decoder weights',
      'zero-initialize additional input weights',
      'identical inner split (blocked; guard; exhaustive-only model selection)',
      'identical batch/site sampling, augmentation, loss, optimizer, LR, training steps, patience',
      'fold-safe normalization fit on training chips only',
      'no site holdout mixing; no outer-label access during selection'],
  modality_arms={'M0':'RGB3 frozen historical baseline',
       'M1':'RGB3+predefined DSM features early fusion',
       'M2':'RGB3+relative four-band MS early fusion'},
  planned_primary_metrics=['paired plot cover MAE difference in pp',
       'cover R2','RMSE','bias','pixel IoU/F1','MOS >40% bias'],
  exclusion='No modality missingness imputation or cross-flight band mosaicking',
  stage1_sensor_registry=str(OUT/'05_harmonized_sensor_registry.csv'),
  stage1_candidate_support=str(OUT/'05_candidate_unit_modality_support.csv'))
target=OUT/'05_optionB_frozen_protocol_STAGE1.json'
target.write_text(json.dumps(protocol,indent=2))
print('SAVED',target)
print('Stage 1 is complete. Do NOT train until exact MS, DSM mask and crown-scale alignment QA is reviewed.')
